In [1]:
# Import Gemini SDK
from google import genai
from google.genai import types

# Initialize Gemini client with your API key
client = genai.Client(api_key="AQ.Ab8RN6L2ebaIjbAqbmvTfEHkmNAwnSdwpKNvzOs_wf9SwNGbmA")

In [3]:
# Helper function to send messages to Gemini
# Accepts a list of messages with roles: system, user, assistant
# Converts them to Gemini's expected format internally
def get_completion_from_messages(messages, temperature=0, max_tokens=500):
    system_instruction = None
    conversation = []

    for msg in messages:
        if msg["role"] == "system":
            # Gemini takes system message separately as system_instruction
            system_instruction = msg["content"]
        else:
            # Gemini uses "model" instead of "assistant"
            role = "model" if msg["role"] == "assistant" else "user"
            conversation.append({
                "role": role,
                "parts": [{"text": msg["content"]}]
            })

    response = client.models.generate_content(
        model="models/gemini-2.5-flash",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=temperature,
            max_output_tokens=max_tokens,
        ),
        contents=conversation
    )
    return response.text

In [5]:
###System Message with Chain-of-Thought Steps
# Delimiter used to separate each reasoning step in the prompt
delimiter = "####"

# System message instructs Gemini to follow a structured 5-step reasoning process
# before responding to the customer query about products
system_message = f"""
Follow these steps to answer the customer queries.
The customer query will be delimited with four hashtags,\
i.e. {delimiter}. 

Step 1:{delimiter} First decide whether the user is \
asking a question about a specific product or products. \
Product cateogry doesn't count. 

Step 2:{delimiter} If the user is asking about \
specific products, identify whether \
the products are in the following list.
All available products: 
1. Product: TechPro Ultrabook
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-UB100
   Warranty: 1 year
   Rating: 4.5
   Features: 13.3-inch display, 8GB RAM, 256GB SSD, Intel Core i5 processor
   Description: A sleek and lightweight ultrabook for everyday use.
   Price: $799.99

2. Product: BlueWave Gaming Laptop
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-GL200
   Warranty: 2 years
   Rating: 4.7
   Features: 15.6-inch display, 16GB RAM, 512GB SSD, NVIDIA GeForce RTX 3060
   Description: A high-performance gaming laptop for an immersive experience.
   Price: $1199.99

3. Product: PowerLite Convertible
   Category: Computers and Laptops
   Brand: PowerLite
   Model Number: PL-CV300
   Warranty: 1 year
   Rating: 4.3
   Features: 14-inch touchscreen, 8GB RAM, 256GB SSD, 360-degree hinge
   Description: A versatile convertible laptop with a responsive touchscreen.
   Price: $699.99

4. Product: TechPro Desktop
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-DT500
   Warranty: 1 year
   Rating: 4.4
   Features: Intel Core i7 processor, 16GB RAM, 1TB HDD, NVIDIA GeForce GTX 1660
   Description: A powerful desktop computer for work and play.
   Price: $999.99

5. Product: BlueWave Chromebook
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-CB100
   Warranty: 1 year
   Rating: 4.1
   Features: 11.6-inch display, 4GB RAM, 32GB eMMC, Chrome OS
   Description: A compact and affordable Chromebook for everyday tasks.
   Price: $249.99

Step 3:{delimiter} If the message contains products \
in the list above, list any assumptions that the \
user is making in their \
message e.g. that Laptop X is bigger than \
Laptop Y, or that Laptop Z has a 2 year warranty.

Step 4:{delimiter} If the user made any assumptions, \
figure out whether the assumption is true based on your \
product information. 

Step 5:{delimiter} First, politely correct the \
customer's incorrect assumptions if applicable. \
Only mention or reference products in the list of \
5 available products, as these are the only 5 \
products that the store sells. \
Answer the customer in a friendly tone.

Use the following format:
Step 1:{delimiter} <step 1 reasoning>
Step 2:{delimiter} <step 2 reasoning>
Step 3:{delimiter} <step 3 reasoning>
Step 4:{delimiter} <step 4 reasoning>
Response to user:{delimiter} <response to customer>

Make sure to include {delimiter} to separate every step.
"""

In [7]:
##Query 1: Price Comparison
# User asks a price comparison question with an incorrect assumption
# (BlueWave Chromebook $249.99 is actually CHEAPER than TechPro Desktop $999.99)
# Gemini should detect and correct this assumption in its chain-of-thought

user_message = f"""
by how much is the BlueWave Chromebook more expensive \
than the TechPro Desktop"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user",   "content": f"{delimiter}{user_message}{delimiter}"},
]

response = get_completion_from_messages(messages)
print("── Query 1: Price Comparison ──")
print(response)

── Query 1: Price Comparison ──
Step 1:#### The user is asking a question about two specific products: "BlueWave Chromebook" and "TechPro Desktop".
Step 2:#### Both "BlueWave Chromebook" and "TechPro Desktop" are in the list of available products.
Step 3:#### The user is assuming that the BlueWave Chromebook is more expensive than the TechPro Desktop.
Step 4:#### The assumption is incorrect. The BlueWave Chromebook costs $249.99, while the TechPro Desktop costs $999.99. The TechPro Desktop is more expensive than the BlueWave Chromebook.
Response to user:#### It seems there might be a slight misunderstanding! The TechPro Desktop is actually more expensive than the BlueWave Chromebook. The TechPro Desktop is priced at $999.99, and the BlueWave Chromebook is $249.99. This means the TechPro Desktop is $750.00 more expensive than the BlueWave Chromebook.


In [8]:
## Query 2: Out-of-Scope Product
# User asks about TVs which are not in the product catalog
# Gemini should follow the steps and inform the user politely
# that only the 5 listed products are available

user_message = f"""
do you sell tvs"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user",   "content": f"{delimiter}{user_message}{delimiter}"},
]

response = get_completion_from_messages(messages)
print("── Query 2: Out-of-Scope Product ──")
print(response)

── Query 2: Out-of-Scope Product ──
Step 1:#### The user is asking a question about a product category (TVs), not a specific product from the provided list.
Step 2:#### The user is not asking about specific products from the list.
Step 3:#### The user is not making any assumptions about specific products.
Step 4:#### N/A
Response to user:#### We currently specialize in computers and laptops! We don't sell TVs at the moment. Would you like to know more about our available laptops or desktops?


In [9]:
### Inner Monologue: Hide Chain-of-Thought
# The full response contains all reasoning steps separated by the delimiter
# We only want to show the final customer-facing reply, not the internal reasoning
# Splitting by delimiter and taking the last chunk extracts just the final response

try:
    # Split on delimiter and take the last segment = "Response to user" section
    final_response = response.split(delimiter)[-1].strip()
except Exception as e:
    # Fallback message if something goes wrong during parsing
    final_response = "Sorry, I'm having trouble right now, please try asking another question."

print("── Final Response to Customer (Inner Monologue Hidden) ──")
print(final_response)

── Final Response to Customer (Inner Monologue Hidden) ──
We currently specialize in computers and laptops! We don't sell TVs at the moment. Would you like to know more about our available laptops or desktops?
